# CT Scanner Simulator

**Group:**
* Jakub Biernat 160248
* Eryk Masian 160228

**Tomograph Model:**
* **Cone-beam**

**Technologies Used:**
* **Language:** Python
* **Libraries:** numpy, matplotlib, ipywidgets, pydicom, pillow

In [2]:
# ---------------- IMPORTY ----------------
import os
import sys
import io

import numpy as np
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import ipywidgets as widgets
from PIL import Image

from src.radon_transform import get_sinogram, inverse_radon
from src.emiters import emitter_pos
from src.filter import apply_filter
from src.rmse import calculate_rmse
from src.handle_dicoms import load_image, save_dicom
from src.detectors import detectors_pos

sys.path.append(os.path.abspath('..'))

# ---------------- OBRAZY ----------------
available_images = {
    "Shepp_logan": "../data/input/Shepp_logan.jpg",
    "CT_ScoutView": "../data/input/CT_ScoutView.jpg",
    "SADDLE_PE": "../data/input/SADDLE_PE.jpg",
    "CT_ScoutView-large": "../data/input/CT_ScoutView-large.jpg",
    "Kolo": "../data/input/Kolo.jpg",
    "Kropka": "../data/input/Kropka.jpg",
    "Kwadraty2": "../data/input/Kwadraty2.jpg",
    "Paski2": "../data/input/Paski2.jpg",
    "SADDLE_PE-large": "../data/input/SADDLE_PE-large.jpg",
    "CT_ScoutView_DCM": "../data/input/CT_ScoutView.dcm",
    "CT_ScoutView-large_DCM": "../data/input/CT_ScoutView-large.dcm",
    "Kolo_DCM": "../data/input/Kolo.dcm",
    "Kropka_DCM": "../data/input/Kropka.dcm",
    "Kwadraty2_DCM": "../data/input/Kwadraty2.dcm",
    "Paski2_DCM": "../data/input/Paski2.dcm",
    "SADDLE_PE_DCM": "../data/input/SADDLE_PE.dcm",
    "SADDLE_PE-large_DCM": "../data/input/SADDLE_PE-large.dcm",
    "Shepp_logan_DCM": "../data/input/Shepp_logan.dcm"
}

# ---------------- STAŁE I ZMIENNE GLOBALNE ----------------
PREVIEW_FIGSIZE = (6, 6)
PREVIEW_DPI = 150
PREVIEW_EMITTER_POS = 90  # kąt emitera do preview
image = None
info = None
reconstruction_history = None
full_sinogram = None
latest_reconstructed_image = None

# ---------------- UI ----------------
#Slidery/buttony/checkboxy/progress/bar
image_selector = widgets.Dropdown(
    options=list(available_images.keys()),
    value=list(available_images.keys())[0],
    description="Obraz:"
)

detectors_slider = widgets.IntSlider(
    value=90, min=30, max=720, step=30,
    description="Detektory:"
)

span_slider = widgets.FloatSlider(
    value=30, min=15, max=270, step=15,
    description="Rozpiętość:"
)

scans_slider = widgets.IntSlider(
    value=360,
    min=90,
    max=720,
    step=90,
    description="Skany:"
)

history_slider = widgets.IntSlider(
    value=180,
    min=1,
    max=180,
    step=1,
    description='Iteracja:'
)

filter_checkbox = widgets.Checkbox(
    value=False,
    description='Użyj filtru splotowego'
)

run_button = widgets.Button(
    description="Uruchom Skanowanie",
    button_style='info'
)

progress_bar = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description='Postęp:',
    bar_style='info'
)

save_button = widgets.Button(
    description="Zapisz DICOM",
    button_style='success'
)

#Formularz edycji danych do zapisu dicom
style = {'description_width': 'initial'}
patient_name_input = widgets.Text(description="PatientName", style=style)
patient_id_input = widgets.Text(description="PatientID", style=style)
patient_birth_date_input = widgets.Text(description="PatientBirthDate", style=style)
patient_sex_input = widgets.Text(description="PatientSex", style=style)
patient_weight_input = widgets.Text(description="PatientWeight", style=style)
patient_size_input = widgets.Text(description="PatientSize", style=style)
study_date_input = widgets.Text(description="StudyDate", style=style)
study_description_input = widgets.Textarea(description="StudyDescription", style=style)
filename_input = widgets.Text(
    value="Zrekonstruowany_DICOM.dcm",
    description="Filename",
    style=style
)

#Labele/image/output
alpha_label = widgets.Label()
preview_canvas = widgets.Image(format="png")
info_out = widgets.Output()
results_out = widgets.Output()

preview_canvas.layout = widgets.Layout(
    width="300px",
    height="300px"
)

left_panel = widgets.VBox([
    image_selector,
    detectors_slider,
    span_slider,
    widgets.HBox([scans_slider, alpha_label]),
    run_button,
    filter_checkbox,
], layout=widgets.Layout(width="30%"))

middle_panel = widgets.VBox([
    preview_canvas
], layout=widgets.Layout(
    width="40%",
    align_items="center",
    justify_content="center"
))

right_panel = widgets.VBox([
    info_out
], layout=widgets.Layout(width="30%"))

ui = widgets.HBox([
    left_panel,
    middle_panel,
    right_panel
])

#Formularz pacjenta
dicom_form = widgets.VBox([
    patient_name_input,
    patient_id_input,
    patient_birth_date_input,
    patient_sex_input,
    patient_weight_input,
    patient_size_input,
    study_date_input,
    study_description_input,
    filename_input
])

scan_ui = widgets.VBox([history_slider]) #UI po dokonaniu skanu

display(ui)

# ---------------- FUNKCJE POMOCNICZE ----------------
#Zamienia wykres matplotlib na png (żeby nie mrugało)
def fig_to_png(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=PREVIEW_DPI)
    buf.seek(0)
    return buf.read()

#Funkcja skanowania
def execute_scan(b):
    global full_sinogram, reconstruction_history, latest_reconstructed_image

    with results_out:
        clear_output(wait=True)
        detectors = detectors_slider.value
        span = span_slider.value
        scans = scans_slider.value
        use_filter = filter_checkbox.value

        scan_ui.layout.display = 'none'
        progress_bar.value = 10
        display(progress_bar)

        progress_bar.description = 'Sinogram...'
        full_sinogram = get_sinogram(image, detectors, scans, span)
        progress_bar.value = 40

        progress_bar.description = 'Filtrowanie...'
        if use_filter:
            sinogram_to_reconstruct = apply_filter(full_sinogram)
        else:
            sinogram_to_reconstruct = full_sinogram
        progress_bar.value = 50

        progress_bar.description = 'Rekonstrukcja...'
        reconstruction_history = inverse_radon(sinogram_to_reconstruct, image.shape, detectors, scans, span, return_history=True)
        progress_bar.value = 90

        latest_reconstructed_image = reconstruction_history[-1]

        progress_bar.value = 100
        progress_bar.description = 'Gotowe!'
        history_slider.max = scans
        history_slider.value = scans
        scan_ui.layout.display = 'block'

        #Wypełnienie formularza domyślnymi danymi
        patient_name_input.value = str(info.get("PatientName", ""))
        patient_id_input.value = str(info.get("PatientID", ""))
        patient_birth_date_input.value = str(info.get("PatientBirthDate", ""))
        patient_sex_input.value = str(info.get("PatientSex", ""))
        patient_weight_input.value = str(info.get("PatientWeight", ""))
        patient_size_input.value = str(info.get("PatientSize", ""))
        study_date_input.value = str(info.get("StudyDate", ""))
        study_description_input.value = str(info.get("StudyDescription", ""))

        display_scan_ui()

#Zapis pliku jako dicom
def save_current_dicom(b):
    global latest_reconstructed_image, info
    print("Trwa zapisywanie...")

    new_info = info.copy()
    new_info["PatientName"] = patient_name_input.value
    new_info["PatientID"] = patient_id_input.value
    new_info["PatientBirthDate"] = patient_birth_date_input.value
    new_info["PatientSex"] = patient_sex_input.value
    new_info["PatientWeight"] = patient_weight_input.value
    new_info["PatientSize"] = patient_size_input.value
    new_info["StudyDate"] = study_date_input.value
    new_info["StudyDescription"] = study_description_input.value
    filename = filename_input.value

    image_to_save = latest_reconstructed_image
    image_to_save = np.clip(image_to_save, 0, 255).astype(np.uint8)

    save_dicom(image_to_save, new_info, filename)

    print(f"Zapisano: {filename}")


# ---------- WYŚWIETLANIE SCAN UI -------------
def display_scan_ui():
    clear_output(wait=True)
    image_widget = widgets.Image(format='png', width=900, height=350)

    plt.ioff()
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(image, cmap='gray')
    axes[0].set_title("Obraz wejściowy")
    axes[0].axis('off')

    im_sino = axes[1].imshow(np.zeros_like(full_sinogram), cmap='gray', aspect='auto', vmin=0, vmax=np.max(full_sinogram))
    axes[1].set_title("Sinogram (Tworzenie)")
    axes[1].axis('off')

    im_recon = axes[2].imshow(np.zeros_like(image), cmap='gray', vmin=0, vmax=255)
    axes[2].set_title("Rekonstrukcja")
    axes[2].axis('off')
    plt.tight_layout(rect=[0, 0, 1, 0.92])

    #Odświeżanie klatek, do rekonstrukcji po iteracji
    def update_frame(change):
        step = history_slider.value

        partial_sinogram = np.zeros_like(full_sinogram)
        partial_sinogram[:step, :] = full_sinogram[:step, :]

        im_sino.set_data(partial_sinogram)
        im_recon.set_data(reconstruction_history[step - 1])

        current_rmse = calculate_rmse(image, reconstruction_history[step - 1])
        fig.suptitle(f"Krok: {step}/{history_slider.max} | Ostatnie RMSE: {current_rmse:.4f}", fontsize=14)

        buf = io.BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight')
        buf.seek(0)
        image_widget.value = buf.read()
        buf.close()

    history_slider.observe(update_frame, names='value')
    display(widgets.VBox([scan_ui, image_widget, dicom_form, save_button]))
    update_frame(None)

# ---------------- ZMIANA OBRAZU ----------------
def update(*args):
    global image, info
    name = image_selector.value
    path = available_images[name]

    info, image = load_image(path)
    if len(image.shape) == 3:
        image = np.mean(image, axis=2)

    detectors = detectors_slider.value
    span = span_slider.value

    xe, ye = emitter_pos(image, PREVIEW_EMITTER_POS)
    dets = detectors_pos(image, PREVIEW_EMITTER_POS, detectors, span)

    scans = scans_slider.value
    angle = 360 / scans  # delta alpha

    alpha_label.value = f"α = {angle:.3f}°"

    #Rysowanie detektorów i emiterów
    fig, ax = plt.subplots(figsize=PREVIEW_FIGSIZE, dpi=PREVIEW_DPI)

    ax.imshow(image, cmap="gray", aspect="equal")

    # emitter
    ax.scatter(xe, ye, c="red", s=80)

    # detektory + linie
    if len(dets) > 0:
        dx, dy = zip(*dets)
        ax.scatter(dx, dy, c="blue", s=30)

        for x, y in dets:
            ax.plot([xe, x], [ye, y],
                    c="red", linewidth=0.5, alpha=0.6)

    ax.set_title(name)
    ax.axis("off")

    preview_canvas.value = fig_to_png(fig)
    plt.close(fig)

    #Informacje o pacjencie
    with info_out:
        info_out.clear_output(wait=True)
        print("=== INFO DICOM ===")
        for k, v in info.items():
            print(f"{k}: {v}")

# ---------------- OBSERWATORY ----------------
for w in [image_selector, detectors_slider, span_slider, scans_slider, filter_checkbox]:
    w.observe(update, "value")

update()
run_button.on_click(execute_scan)
save_button.on_click(save_current_dicom)
display(results_out)

Output()